# ASL Recognition — V3 Motion-Aware Training Pipeline

This notebook contains the full `train_asl_v3_motion.py` pipeline, split into
labeled cells so each stage of the system can be run, inspected, and explained
independently (data loading → augmentation → model → training → evaluation →
real-time inference).

**Architecture summary:** BiGRU (2 layers) with temporal attention, trained on
MediaPipe landmark sequences (60 frames). The V3 update expands the raw
155-dim per-frame feature vector into a 308-dim vector by appending
frame-to-frame velocity (motion) channels, on top of position and hand
presence flags.

> Run the cells top to bottom. Cell 2 mounts Google Drive since `CFG["data_root"]`
> and `CFG["save_dir"]` point to Drive paths — update these paths for your own setup.

## 0. Mount Google Drive

Optional convenience cell for Colab. The training config (`CFG`) below expects
`data_root` and `save_dir` to live under `/content/drive/MyDrive/...`. Skip this
cell if you're running locally or have already mounted Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & Reproducibility

Standard imports (NumPy, PyTorch, scikit-learn) plus a `set_seed()` helper that
fixes the seed for Python's `random`, NumPy, and PyTorch (CPU + CUDA), and
forces deterministic cuDNN behavior. This is called once at import time so
every run of the notebook starts from the same random state, making results
reproducible.

In [ ]:
import json
import os
import random
from collections import Counter
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
SEED = 42
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

## 2. Configuration (`CFG`)

A single dictionary holding every hyperparameter for the run: data paths,
sequence/feature dimensions (155 raw → 308 after motion expansion), the full
augmentation toggle set, model sizing (GRU hidden sizes, dropout rates),
optimization settings (epochs, batch size, learning rate schedule, early
stopping patience), test-time augmentation (TTA) view count, and the
focal-loss / class-weighting / cluster-penalty knobs used to specifically
target historically confused sign pairs.

In [ ]:
CFG = dict(
    # Paths
    data_root        = "/content/drive/MyDrive/model_videos_proc",
    save_dir         = "/content/drive/MyDrive/modelfolder100_last version5",
    # Data
    seq_len          = 60,
    feat_dim_raw     = 155,   
    feat_dim         = 308,   
    test_size        = 0.15,
    val_size         = 0.15,

    # Augmentation
    aug_time_warp    = True,
    aug_noise        = True,
    aug_scale        = True,
    aug_mirror       = True,
    aug_frame_drop   = True,
    aug_mixup        = True,
    aug_xy_rotate    = True,
    aug_joint_drop   = True,
    aug_time_shift   = True,
    noise_std        = 0.008,
    scale_range      = (0.85, 1.15),
    time_warp_max    = 4,
    frame_drop_p     = 0.05,
    joint_drop_p     = 0.05,
    time_shift_max   = 5,
    xy_rotate_max    = 10.0,
    mixup_alpha      = 0.3,
    mixup_prob       = 0.5,

    # Model  (V3 architecture)
    gru1_hidden      = 256,
    gru2_hidden      = 128,
    dropout_gru      = 0.35,
    dropout_cls1     = 0.40,
    dropout_cls2     = 0.30,

    # Training
    epochs           = 80,
    batch_size       = 32,
    lr               = 3e-4,
    max_lr           = 1e-3,
    weight_decay     = 1e-4,
    patience         = 20,
    label_smooth     = 0.1,
    grad_clip        = 1.0,

    # TTA
    tta_n            = 5,

    # Focal loss
    focal_gamma          = 2.0,
    target_class_weight  = 2.0,
    cluster_penalty      = 2.0,
)

## 3. Feature Layout Constants

Column-index bookkeeping for the two feature layouts used in the pipeline:

- **Raw on-disk layout (155 dims):** 63 left-hand + 63 right-hand + 27 pose
  coordinates, followed by a left-hand and right-hand presence flag.
- **Post-motion-expansion layout (308 dims):** the same 153 position
  coordinates, followed by 153 velocity (frame-to-frame delta) values, then
  the two presence flags — now relocated to the end of the vector.

`TARGET_SIGNS` lists the specific signs that receive harder augmentation and
extra class weight because they've historically been hardest to classify.
`CONFUSION_CLUSTERS` groups semantically or visually similar signs (plus a few
pairs found empirically from confusion-matrix analysis) so the loss function
can penalize cross-cluster mistakes more heavily than random ones.

In [ ]:

HAND_COLS = slice(0, 126)
LH_COLS   = slice(0, 63)
RH_COLS   = slice(63, 126)
POSE_COLS = slice(126, 153)
COORD_COLS = slice(0, 153)
LH_FLAG   = 153
RH_FLAG   = 154

POS_COLS_M = slice(0, 153)
VEL_COLS_M = slice(153, 306)
LH_FLAG_M  = 306
RH_FLAG_M  = 307
FEAT_DIM_MOTION = 308

TARGET_SIGNS = {
    "deaf", "again", "help", "family", "drink", "my", "no", "understand", "hello",
    "why", "from", "saturday", "tuesday", "brother",
    "please", "sick", "thank", "always", "alarm", "cat", "happy",
}

CONFUSION_CLUSTERS = [
    {"why", "what", "who", "where", "how"},
    {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"},
    {"brother", "boy", "book", "blue", "bye", "because", "before", "buy"},
    {"I", "me", "my", "you", "your", "who"},
    {"after", "before", "next", "always", "again", "will"},
    {"happy", "sad", "angry", "enjoy", "fun", "love", "tired"},
    {"drink", "eat", "coffee", "milk", "apple", "banana"},
    {"my", "please", "happy"},
    {"go", "can"},
]

## 4. Motion Feature Expansion (V3)

`add_motion_features()` is the core V3 addition: it takes a raw `(T, 155)`
landmark sequence and expands it to `(T, 308)` by computing a velocity channel
(frame-to-frame coordinate deltas) alongside the original position channel.
The first frame's velocity is set to zero rather than wrapping around, since
there's no real motion to compute for a non-existent "frame -1".

This must run on raw, un-normalized coordinates, *after* any
trajectory-altering augmentation (so velocity reflects the augmented motion)
but *before* wrist-relative anchoring/normalization.

In [ ]:
def add_motion_features(seq: np.ndarray) -> np.ndarray:
    T = seq.shape[0]
    coords = seq[:, :153]
    flags  = seq[:, 153:155]

    velocity = np.zeros_like(coords)
    velocity[1:] = coords[1:] - coords[:-1]

    out = np.concatenate([coords, velocity, flags], axis=1)
    return out.astype(np.float32)

## 5. Augmentation Helpers

A set of stochastic augmentations, each operating on **raw 155-dim** sequences
(before motion expansion):

- `aug_temporal_noise` — Gaussian jitter on coordinates only.
- `aug_scale` — random isotropic scaling of coordinates.
- `aug_time_warp` — per-frame random temporal offsets, with presence flags
  re-binarized after warping.
- `aug_time_shift` — cyclic shift of the whole sequence in time.
- `aug_xy_rotate` — small random rotation in the XY plane.
- `aug_joint_dropout` — zeroes out random joints to simulate landmark
  detection failures.
- `aug_mirror_hands` — swaps left/right hand blocks and negates x-coordinates
  (used for deterministic mirror-expansion of the dataset, not random
  per-sample augmentation).
- `aug_frame_dropout` — zeroes out entire random frames.
- `mirror_sequence` — a thin wrapper around `aug_mirror_hands` used by TTA and
  real-time inference.
- `augment()` — the stochastic augmentation chain applied during training;
  it applies a random subset of the above with some probability each, and
  gives `TARGET_SIGNS` (historically hard signs) stronger augmentation
  parameters (e.g. wider scale range, longer time warp, higher frame-drop
  probability).

In [ ]:
def aug_temporal_noise(seq: np.ndarray, std: float = 0.005) -> np.ndarray:
  
    out = seq.copy()
    out[:, :153] += np.random.normal(0, std, (seq.shape[0], 153)).astype(np.float32)
    return out


def aug_scale(seq: np.ndarray, scale_range=(0.85, 1.15)) -> np.ndarray:
    s = np.random.uniform(*scale_range)
    out = seq.copy()
    out[:, :153] *= s
    return out


def aug_time_warp(seq: np.ndarray, max_shift: int = 4) -> np.ndarray:
    T = seq.shape[0]
    shifts = np.random.randint(-max_shift, max_shift + 1, size=T).astype(float)
    grid   = np.clip(np.arange(T) + shifts, 0, T - 1).astype(int)
    warped = seq[grid]
    warped[:, LH_FLAG] = (warped[:, LH_FLAG] >= 0.5).astype(np.float32)
    warped[:, RH_FLAG] = (warped[:, RH_FLAG] >= 0.5).astype(np.float32)
    return warped


def aug_time_shift(seq: np.ndarray, max_shift: int = 5) -> np.ndarray:
    shift = np.random.randint(-max_shift, max_shift + 1)
    out   = np.roll(seq, shift, axis=0)
    out[:, LH_FLAG] = (out[:, LH_FLAG] >= 0.5).astype(np.float32)
    out[:, RH_FLAG] = (out[:, RH_FLAG] >= 0.5).astype(np.float32)
    return out


def aug_xy_rotate(seq: np.ndarray, max_deg: float = 10.0) -> np.ndarray:
    angle = np.radians(np.random.uniform(-max_deg, max_deg))
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    out   = seq.copy()
    coords = out[:, :153].reshape(seq.shape[0], 51, 3)
    x_new  = coords[:, :, 0] * cos_a - coords[:, :, 1] * sin_a
    y_new  = coords[:, :, 0] * sin_a + coords[:, :, 1] * cos_a
    coords[:, :, 0] = x_new
    coords[:, :, 1] = y_new
    out[:, :153] = coords.reshape(seq.shape[0], 153)
    return out


def aug_joint_dropout(seq: np.ndarray, p: float = 0.05) -> np.ndarray:
    out = seq.copy()
    n_joints = 51
    mask = np.random.rand(n_joints) < p
    for j in np.where(mask)[0]:
        out[:, j*3 : j*3+3] = 0.0
    return out


def aug_mirror_hands(seq: np.ndarray) -> np.ndarray:
    out     = seq.copy()
    lh      = seq[:, LH_COLS].copy()
    rh      = seq[:, RH_COLS].copy()
    lh_flag = seq[:, LH_FLAG].copy()
    rh_flag = seq[:, RH_FLAG].copy()

    out[:, LH_COLS] = rh
    out[:, RH_COLS] = lh
    out[:, LH_FLAG] = rh_flag
    out[:, RH_FLAG] = lh_flag

    for col_slice in [LH_COLS, RH_COLS]:
        block = out[:, col_slice].reshape(seq.shape[0], 21, 3)
        block[:, :, 0] *= -1
        out[:, col_slice] = block.reshape(seq.shape[0], -1)

    pose = out[:, POSE_COLS].reshape(seq.shape[0], 9, 3)
    pose[:, :, 0] *= -1
    out[:, POSE_COLS] = pose.reshape(seq.shape[0], -1)
    return out


def aug_frame_dropout(seq: np.ndarray, p: float = 0.05) -> np.ndarray:
    out  = seq.copy()
    mask = np.random.rand(seq.shape[0]) < p
    out[mask] = 0.0
    return out


def mirror_sequence(seq: np.ndarray) -> np.ndarray:
    return aug_mirror_hands(seq.copy())


def augment(seq: np.ndarray, cfg: dict, label: str = "") -> np.ndarray:
    hard = label in TARGET_SIGNS

    if cfg["aug_noise"] and random.random() < 0.5:
        seq = aug_temporal_noise(seq, cfg["noise_std"])

    if cfg["aug_scale"] and random.random() < 0.5:
        sr = (0.75, 1.25) if hard else cfg["scale_range"]
        seq = aug_scale(seq, sr)

    if cfg.get("aug_xy_rotate", False) and random.random() < 0.4:
        seq = aug_xy_rotate(seq, cfg.get("xy_rotate_max", 10.0))

    if cfg["aug_time_warp"] and random.random() < 0.5:
        tw = 7 if (hard and label in {"again", "help", "why", "from"}) else cfg["time_warp_max"]
        seq = aug_time_warp(seq, tw)

    if cfg.get("aug_time_shift", False) and random.random() < 0.4:
        seq = aug_time_shift(seq, cfg.get("time_shift_max", 5))

    if cfg.get("aug_joint_drop", False) and random.random() < 0.4:
        seq = aug_joint_dropout(seq, cfg.get("joint_drop_p", 0.05))

    if cfg["aug_frame_drop"] and random.random() < 0.4:
        fp = 0.12 if (hard and label in {"drink", "family", "saturday", "tuesday"}) \
             else cfg["frame_drop_p"]
        seq = aug_frame_dropout(seq, fp)

    return seq

## 6. Normalization

`_normalise_coords()` operates on the **308-dim post-motion** sequence and
performs three steps:

1. **Wrist-relative anchoring** — hand coordinates in the position block are
   made relative to the wrist joint (velocity is left untouched here since
   it's already a relative/differenced quantity).
2. **Independent z-score normalization** of the position block and the
   velocity block separately, because velocity magnitudes are typically much
   smaller than position magnitudes — a shared mean/std would let position
   dominate and wash out the velocity signal.
3. Presence flags are passed through untouched.

`_prepare_sequence()` is the single shared pipeline used everywhere a raw
`(T, 155)` on-disk sequence needs to become a model-ready, normalized `(T,
308)` tensor input: mirror → augment (train only) → motion expansion →
normalization. Centralizing this avoids the train/inference paths drifting
out of sync.

In [ ]:
def _normalise_coords(seq: np.ndarray) -> np.ndarray:

    out = seq.copy()

    # ── 1. Wrist-relative anchoring (position block only) ──
    lh = out[:, 0:63].reshape(-1, 21, 3)
    lh = lh - lh[:, 0:1, :]
    out[:, 0:63] = lh.reshape(-1, 63)

    rh = out[:, 63:126].reshape(-1, 21, 3)
    rh = rh - rh[:, 0:1, :]
    out[:, 63:126] = rh.reshape(-1, 63)

    # ── 2a. Normalise position block ──
    pos = out[:, POS_COLS_M]
    pos_mean, pos_std = pos.mean(), pos.std()
    if pos_std > 1e-6:
        out[:, POS_COLS_M] = (pos - pos_mean) / pos_std

    # ── 2b. Normalise velocity block (independently) ──
    vel = out[:, VEL_COLS_M]
    vel_mean, vel_std = vel.mean(), vel.std()
    if vel_std > 1e-6:
        out[:, VEL_COLS_M] = (vel - vel_mean) / vel_std

    return out


def _prepare_sequence(raw_seq: np.ndarray, mirror: bool, train: bool,
                       cfg: dict, label: str = "") -> np.ndarray:

    seq = raw_seq
    if mirror:
        seq = aug_mirror_hands(seq)
    if train:
        seq = augment(seq, cfg, label)
    seq = add_motion_features(seq)
    seq = _normalise_coords(seq)
    return seq

## 7. Dataset Class

`SignDataset` is a PyTorch `Dataset` where each sample is stored as a
`(file_path, is_mirrored)` tuple — mirrored copies are injected ahead of time
by `_expand_with_mirrors()` (see next cell) rather than mirrored randomly at
train time. `__getitem__` loads the raw `.npy` landmark file, cleans NaN/Inf
values, and runs it through the shared `_prepare_sequence()` pipeline
(applying augmentation only when `train=True`).

In [ ]:
class SignDataset(Dataset):
    def __init__(self, samples, labels, label2idx, cfg, train=False):
        self.samples   = samples
        self.targets   = [label2idx[l] for l in labels]
        self.train     = train
        self.cfg       = cfg
        self._idx2name = {v: k for k, v in label2idx.items()}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, is_mirrored = self.samples[idx]

        seq = np.load(path).astype(np.float32)
        seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

        label_name = self._idx2name[self.targets[idx]] if self.train else ""
        seq = _prepare_sequence(seq, mirror=is_mirrored, train=self.train,
                                 cfg=self.cfg, label=label_name)

        x = torch.from_numpy(seq)
        y = torch.tensor(self.targets[idx], dtype=torch.long)
        return x, y

## 8. Data Loading, Expansion & Audit

- `load_dataset()` — scans `data_root` for one subfolder per class and
  collects every `.npy` sample path with its class label.
- `_expand_with_mirrors()` — duplicates each training sample as
  `(path, False)` and `(path, True)`, doubling the effective training set
  size with a deterministic mirrored copy of every sample (rather than
  relying on random per-batch mirroring).
- `audit_dataset()` — sanity-checks the RAW on-disk files (still expected to
  be `(60, 155)`), flagging files that fail to load, contain NaN/Inf, are
  all-zero, or have an unexpected shape.

In [ ]:
def load_dataset(data_root: str):
    root    = Path(data_root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    paths, labels = [], []
    for cls in classes:
        for f in (root / cls).glob("*.npy"):
            paths.append(str(f))
            labels.append(cls)
    return paths, labels, classes


def _expand_with_mirrors(paths, labels):
    expanded_samples, expanded_labels = [], []
    for p, l in zip(paths, labels):
        expanded_samples.append((p, False))
        expanded_samples.append((p, True))
        expanded_labels.append(l)
        expanded_labels.append(l)
    return expanded_samples, expanded_labels


def audit_dataset(paths: list, verbose: bool = True) -> list:
    """Audits RAW on-disk files — still expects (60, 155) shape."""
    bad = []
    for p in paths:
        try:
            seq = np.load(p).astype(np.float32)
        except Exception as e:
            bad.append((p, f"load error: {e}"))
            continue
        if np.isnan(seq).any() or np.isinf(seq).any():
            bad.append((p, "contains NaN/Inf"))
        elif seq[:, :153].std() < 1e-8:
            bad.append((p, "all-zero coordinates"))
        elif seq.shape != (60, 155):
            bad.append((p, f"unexpected shape {seq.shape}"))

    if verbose:
        print(f"\n── Data Audit ──")
        print(f"  {len(bad)}/{len(paths)} problematic files found")
        for p, reason in bad[:20]:
            print(f"  [{reason}] {p}")
        if len(bad) > 20:
            print(f"  … and {len(bad) - 20} more")
        print()
    return bad

## 9. Temporal Attention Module

`TemporalAttention` implements a soft attention pooling layer over a sequence
of GRU hidden states: a small feed-forward network scores each timestep,
scores are softmax-normalized into weights across time, and the weighted sum
collapses `(B, T, D)` into a single `(B, D)` context vector — letting the
model learn which frames matter most for a given sign rather than using a
fixed pooling like mean or last-hidden-state.

In [ ]:
class TemporalAttention(nn.Module):

    def __init__(self, input_dim: int, attn_hidden: int = 128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(input_dim, attn_hidden),
            nn.Tanh(),
            nn.Linear(attn_hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scores  = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        context = (weights * x).sum(dim=1)
        return context

## 10. V3 Model Architecture

`ASLv3Classifier` is the motion-aware model:

```
Input (B, 60, 308)   — [position(153), velocity(153), flags(2)]
→ LayerNorm(308)
→ BiGRU-1 (hidden=256, bidirectional)   → output dim 512
→ Dropout(0.35)
→ BiGRU-2 (hidden=128, bidirectional)   → output dim 256
→ Dropout(0.35)
→ TemporalAttention                      → 256-dim context
→ Flag branch (cols 306–307)              → 64-dim embedding
→ Feature Fusion: concat [256, 64]       → 320-dim
→ Classifier:
      Linear(320→512) → ReLU → BatchNorm1d → Dropout(0.40)
      Linear(512→256) → ReLU → Dropout(0.30)
      Linear(256→num_classes)
```

The hand-presence flags are pulled out and processed through a small side
branch (`flag_proj`) so the network has an explicit signal for whether the
left/right hand was even detected in a given clip, and this is fused with the
attention-pooled temporal context before the final classifier head. Weight
init uses orthogonal init for GRU recurrent weights and Xavier for input
weights. `_build_model()` is a small factory that reads the relevant keys out
of `CFG` with safe fallbacks, and `BiGRUClassifier` is kept as a backward
compatible alias for the class name.

In [ ]:
class ASLv3Classifier(nn.Module):


    def __init__(self,
                 feat_dim: int,
                 num_classes: int,
                 gru1_hidden: int = 256,
                 gru2_hidden: int = 128,
                 dropout_gru: float = 0.35,
                 dropout_cls1: float = 0.40,
                 dropout_cls2: float = 0.30):
        super().__init__()

        gru1_out = gru1_hidden * 2
        gru2_out = gru2_hidden * 2
        fused_dim = gru2_out + 64

        self.input_norm = nn.LayerNorm(feat_dim, eps=1e-5)

        self.gru1 = nn.GRU(
            input_size    = feat_dim,
            hidden_size   = gru1_hidden,
            num_layers    = 1,
            batch_first   = True,
            bidirectional = True,
        )
        self.drop1 = nn.Dropout(dropout_gru)

        self.gru2 = nn.GRU(
            input_size    = gru1_out,
            hidden_size   = gru2_hidden,
            num_layers    = 1,
            batch_first   = True,
            bidirectional = True,
        )
        self.drop2 = nn.Dropout(dropout_gru)

        self.attention = TemporalAttention(input_dim=gru2_out, attn_hidden=128)

        self.flag_proj = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_cls1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_cls2),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for name, param in self.gru1.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
        for name, param in self.gru2.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, 308)

        # ── Presence flags now live at the END (cols 306:308) ──
        flags = x[:, :, LH_FLAG_M:RH_FLAG_M + 1]        # (B, T, 2)
        flag_summary = flags.mean(dim=1)                 # (B, 2)
        flag_emb = self.flag_proj(flag_summary)           # (B, 64)

        x = self.input_norm(x)                            # (B, T, 308)

        out1, _ = self.gru1(x)
        out1    = self.drop1(out1)

        out2, _ = self.gru2(out1)
        out2    = self.drop2(out2)

        context = self.attention(out2)                    # (B, 256)

        fused = torch.cat([context, flag_emb], dim=1)     # (B, 320)

        return self.classifier(fused)


BiGRUClassifier = ASLv3Classifier   # backward-compat alias


def _build_model(cfg: dict, num_classes: int) -> "ASLv3Classifier":
    """Factory that reads V3 keys from CFG, with safe fallbacks."""
    return ASLv3Classifier(
        feat_dim     = cfg.get("feat_dim", FEAT_DIM_MOTION),
        num_classes  = num_classes,
        gru1_hidden  = cfg.get("gru1_hidden",  256),
        gru2_hidden  = cfg.get("gru2_hidden",  128),
        dropout_gru  = cfg.get("dropout_gru",  0.35),
        dropout_cls1 = cfg.get("dropout_cls1", 0.40),
        dropout_cls2 = cfg.get("dropout_cls2", 0.30),
    )

## 11. Cluster Penalty Matrix Builder

`build_cluster_penalty_matrix()` builds a `C x C` matrix (C = number of
classes) initialized to 1.0 everywhere, and sets the off-diagonal entries to
`penalty` (default 2.0) for every pair of classes that appear together in the
same `CONFUSION_CLUSTERS` group. This matrix is later used inside the loss
function to specifically penalize the model more when it confuses two signs
from the same known-confusable cluster.

In [ ]:
def build_cluster_penalty_matrix(classes: list,
                                 clusters: list,
                                 penalty: float = 2.0) -> torch.Tensor:
    C        = len(classes)
    M        = torch.ones(C, C)
    name2idx = {c: i for i, c in enumerate(classes)}
    for cluster in clusters:
        idxs = [name2idx[c] for c in cluster if c in name2idx]
        for i in idxs:
            for j in idxs:
                if i != j:
                    M[i][j] = penalty
    return M

## 12. Loss Function — Focal + Label Smoothing + Class Weights + Cluster Penalty

`FocalLabelSmoothingCE` combines four techniques into one custom loss:

1. **Label smoothing** — softens one-hot targets so the model isn't pushed to
   output overconfident probabilities.
2. **Focal weighting** — down-weights easy, already-well-classified examples
   `(1 - p_true) ** gamma` so the model focuses more on hard examples.
3. **Class weights** — extra weight for `TARGET_SIGNS` (the historically
   hardest signs).
4. **Cluster penalty** — an additional multiplier looked up from the
   `cluster_matrix` built above, based on the *true* label and the model's
   *predicted* label, so confusions between known-similar signs are penalized
   more than confusions between unrelated signs.

It also supports soft-label targets (used during Mixup) by falling back to
plain weighted cross-entropy when `targets` isn't already a 1-D class-index
tensor.

In [ ]:
class FocalLabelSmoothingCE(nn.Module):
    def __init__(self, num_classes: int, smoothing: float = 0.1,
                 gamma: float = 2.0,
                 class_weights: torch.Tensor = None,
                 cluster_penalty_matrix: torch.Tensor = None):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.gamma       = gamma
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None
            else torch.ones(num_classes))
        self.register_buffer(
            "cluster_matrix",
            cluster_penalty_matrix if cluster_penalty_matrix is not None
            else torch.ones(num_classes, num_classes))

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = log_probs.exp()

        if targets.dim() == 1:
            smooth_val  = self.smoothing / (self.num_classes - 1)
            true_val    = 1.0 - self.smoothing
            soft_labels = torch.full_like(log_probs, smooth_val)
            soft_labels.scatter_(1, targets.unsqueeze(1), true_val)

            p_true        = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
            focal_weight  = (1 - p_true).pow(self.gamma)
            sample_weight = self.class_weights[targets] * focal_weight

            pred_classes  = logits.argmax(dim=1)
            cluster_w     = self.cluster_matrix[targets, pred_classes]
            sample_weight = sample_weight * cluster_w
        else:
            soft_labels   = targets
            sample_weight = torch.ones(logits.size(0), device=logits.device)

        per_sample = -(soft_labels * log_probs).sum(dim=-1)
        return (per_sample * sample_weight).mean()

## 13. Sampler & Mixup

- `make_weighted_sampler()` — builds a `WeightedRandomSampler` keyed on
  `(class, is_mirrored)` pairs so that rarer combinations are sampled more
  often, balancing both class frequency and mirror/non-mirror ratio within a
  batch.
- `mixup_batch()` — standard Mixup: linearly interpolates two random samples
  in a batch (and their one-hot labels) using a Beta-distributed mixing
  coefficient `lam`. After interpolating, the presence-flag columns (which
  must stay binary) are re-binarized rather than left as blended floats.

In [ ]:
def make_weighted_sampler(targets, mirror_flags):
    keys    = [(t, int(m)) for t, m in zip(targets, mirror_flags)]
    counts  = Counter(keys)
    weights = [1.0 / counts[k] for k in keys]
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(weights),
        num_samples = len(targets),
        replacement = True,
    )


def mixup_batch(x: torch.Tensor, y: torch.Tensor,
                num_classes: int, alpha: float = 0.3):
    lam = np.random.beta(alpha, alpha)
    B   = x.size(0)
    idx = torch.randperm(B, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[idx]

    # Re-binarize flag columns after mixup interpolation (now at 306/307)
    mixed_x[:, :, LH_FLAG_M] = (mixed_x[:, :, LH_FLAG_M] >= 0.5).float()
    mixed_x[:, :, RH_FLAG_M] = (mixed_x[:, :, RH_FLAG_M] >= 0.5).float()

    y_a    = F.one_hot(y, num_classes).float()
    y_b    = F.one_hot(y[idx], num_classes).float()
    soft_y = lam * y_a + (1 - lam) * y_b

    return mixed_x, soft_y

## 14. Training Step

`train_one_epoch()` runs one full pass over the training loader. For each
batch it optionally applies Mixup — with a special "target-aware" variant
that, when at least 2 samples in the batch belong to `TARGET_SIGNS`,
preferentially mixes hard-sign samples with each other (using a higher alpha,
i.e. blending strength) rather than mixing them with easy classes, so the
hardest signs get extra augmentation pressure without being diluted by easy
ones. It then computes the loss, skips (and counts) any batch that produces
NaN/Inf loss, backprops with gradient clipping, and steps both the optimizer
and the OneCycle LR scheduler.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler,
                    criterion, device, grad_clip, cfg,
                    num_classes, classes):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    nan_batches = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        use_mixup = cfg["aug_mixup"] and (random.random() < cfg["mixup_prob"])
        if use_mixup:
            target_mask = torch.tensor(
                [classes[yi.item()] in TARGET_SIGNS for yi in y],
                device=device)

            if target_mask.sum() >= 2:
                target_idx  = target_mask.nonzero(as_tuple=True)[0]
                other_idx   = (~target_mask).nonzero(as_tuple=True)[0]
                perm_target = target_idx[torch.randperm(len(target_idx), device=device)]
                perm_other  = other_idx[torch.randperm(len(other_idx), device=device)] \
                              if len(other_idx) > 0 else other_idx
                perm = torch.empty(len(y), dtype=torch.long, device=device)
                perm[target_idx] = perm_target
                if len(other_idx) > 0:
                    perm[other_idx] = perm_other
                alpha   = cfg["mixup_alpha"] * 1.5
                lam     = np.random.beta(alpha, alpha)
                mixed_x = lam * x + (1 - lam) * x[perm]
                mixed_x[:, :, LH_FLAG_M] = (mixed_x[:, :, LH_FLAG_M] >= 0.5).float()
                mixed_x[:, :, RH_FLAG_M] = (mixed_x[:, :, RH_FLAG_M] >= 0.5).float()
                y_a     = F.one_hot(y, num_classes).float()
                y_b     = F.one_hot(y[perm], num_classes).float()
                x_in, y_in = mixed_x, lam * y_a + (1 - lam) * y_b
            else:
                x_in, y_in = mixup_batch(x, y, num_classes, cfg["mixup_alpha"])
        else:
            x_in, y_in = x, y

        optimizer.zero_grad()
        logits = model(x_in)
        loss   = criterion(logits, y_in)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            if nan_batches <= 5:
                print(f"  [WARNING] NaN/Inf loss — skipping batch. "
                      f"x: min={x.min():.3f} max={x.max():.3f} std={x.std():.3f}")
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * x.size(0)
        with torch.no_grad():
            if use_mixup:
                correct += (model(x).argmax(1) == y).sum().item()
            else:
                correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    if nan_batches:
        print(f"  [WARNING] {nan_batches} NaN/Inf batches skipped this epoch.")

    if total == 0:
        return float("nan"), 0.0
    return total_loss / total, correct / total

## 15. Evaluation

`evaluate()` runs a no-grad forward pass over a data loader (validation or
test), accumulating loss and accuracy while skipping any NaN/Inf-loss batches,
and returns the predictions/targets alongside the aggregate metrics for
downstream reporting (classification report, confusion matrix, etc.).

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []

    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        total_loss += loss.item() * x.size(0)
        preds       = logits.argmax(1)
        correct    += (preds == y).sum().item()
        total      += x.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    if total == 0:
        return float("nan"), 0.0, all_preds, all_targets
    return total_loss / total, correct / total, all_preds, all_targets

## 16. Test-Time Augmentation (Hand-Agnostic, Motion-Aware)

`predict_tta()` builds several "views" of a single raw `(T, 155)` sequence and
averages the model's softmax predictions across them:

- View 1: the clean sequence, unmodified.
- View 2: the mirrored sequence (always included, since the model should be
  hand-agnostic).
- Views 3..n: mildly augmented versions (noise, scale, time-warp/shift) of the
  raw sequence, with motion features and normalization applied *after* the
  perturbation so velocity correctly reflects the perturbed trajectory.

`evaluate_tta()` runs this TTA procedure over an entire `SignDataset` (loading
each raw file from disk) and reports overall TTA accuracy plus predictions and
targets for further analysis.

In [ ]:
@torch.no_grad()
def predict_tta(model, x_np: np.ndarray, cfg: dict, device,
                n: int = 5) -> torch.Tensor:

    model.eval()
    views = []

    clean = _prepare_sequence(x_np.copy(), mirror=False, train=False, cfg=cfg)
    views.append(torch.from_numpy(clean).unsqueeze(0).to(device))

    mirrored = _prepare_sequence(x_np.copy(), mirror=True, train=False, cfg=cfg)
    views.append(torch.from_numpy(mirrored).unsqueeze(0).to(device))

    tta_cfg = dict(cfg,
                   aug_noise=True,      aug_scale=True,
                   aug_time_warp=True,  aug_mirror=False,
                   aug_frame_drop=False, aug_mixup=False,
                   aug_xy_rotate=False,  aug_joint_drop=False,
                   aug_time_shift=True,
                   noise_std=0.003,     scale_range=(0.92, 1.08),
                   time_warp_max=2,     frame_drop_p=0.0,
                   time_shift_max=2)

    for _ in range(max(0, n - 2)):
        raw_aug = augment(x_np.copy(), tta_cfg)
        view = add_motion_features(raw_aug)
        view = _normalise_coords(view)
        views.append(torch.from_numpy(view).unsqueeze(0).to(device))

    batch  = torch.cat(views, dim=0)
    logits = model(batch)
    probs  = torch.softmax(logits, dim=-1).mean(dim=0)
    return probs


@torch.no_grad()
def evaluate_tta(model, test_ds, cfg, device, n_views: int = 5):
    """Full TTA evaluation over a SignDataset. Loads RAW files from disk."""
    model.eval()
    correct, total = 0, 0
    all_preds, all_targets = [], []

    for idx in range(len(test_ds)):
        path, _ = test_ds.samples[idx]
        raw     = np.load(path).astype(np.float32)
        raw     = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
        probs   = predict_tta(model, raw, cfg, device, n=n_views)
        pred    = probs.argmax().item()
        label   = test_ds.targets[idx]
        correct += int(pred == label)
        total   += 1
        all_preds.append(pred)
        all_targets.append(label)

    return correct / total, all_preds, all_targets

## 17. Confusion Analysis

`print_top_confusions()` builds a confusion matrix from targets/predictions,
zeroes out the diagonal (correct predictions), and prints the top-K most
frequent misclassification pairs (true sign → predicted sign) — the main
diagnostic tool used to identify which signs need new `CONFUSION_CLUSTERS`
entries or harder augmentation.

In [ ]:
def print_top_confusions(targets, preds, classes, top_k: int = 10):
    cm = confusion_matrix(targets, preds)
    np.fill_diagonal(cm, 0)
    confused = []
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j] > 0:
                confused.append((cm[i, j], classes[i], classes[j]))
    confused.sort(reverse=True)

    print(f"\n── Top-{top_k} confused pairs (true → predicted) ──")
    for count, true_cls, pred_cls in confused[:top_k]:
        print(f"  {true_cls:<15} → {pred_cls:<15}  ({count} times)")
    print()

## 18. Main Training Loop

`train(cfg)` ties everything together end to end:

1. Loads and audits the dataset, splits into train/val/test (stratified).
2. Expands the training split with deterministic mirrored copies.
3. Builds the datasets/loaders, with a weighted sampler on the training set.
4. Builds the `ASLv3Classifier` model, per-class weights, and the cluster
   penalty matrix, then assembles the `FocalLabelSmoothingCE` loss.
5. Trains with `AdamW` + a `OneCycleLR` schedule, tracking best validation
   accuracy with early stopping (`patience`).
6. Reloads the best checkpoint and runs a full evaluation: validation
   confusion analysis, standard test evaluation, and TTA test evaluation,
   printing classification reports and top confusions for each.
7. Saves the best model checkpoint and the `label2idx` mapping to
   `cfg["save_dir"]`.

This is the cell that actually launches training when run.

In [ ]:
def train(cfg: dict):
    set_seed()
    os.makedirs(cfg["save_dir"], exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    paths, labels, classes = load_dataset(cfg["data_root"])
    num_classes = len(classes)
    label2idx   = {c: i for i, c in enumerate(classes)}
    print(f"Classes ({num_classes}): {classes}")
    print(f"Total samples (before mirror expansion): {len(paths)}")
    print(f"Model feat_dim (with motion channels): {cfg.get('feat_dim', FEAT_DIM_MOTION)}")

    cfg = dict(cfg, classes=classes)

    audit_dataset(paths, verbose=True)

    # ── Train / val / test split ──
    tr_paths, te_paths, tr_labels, te_labels = train_test_split(
        paths, labels, test_size=cfg["test_size"],
        stratify=labels, random_state=SEED)
    val_rel = cfg["val_size"] / (1 - cfg["test_size"])
    tr_paths, va_paths, tr_labels, va_labels = train_test_split(
        tr_paths, tr_labels, test_size=val_rel,
        stratify=tr_labels, random_state=SEED)

    # ── Expand train with mirrored copies ──
    tr_samples, tr_labels_exp = _expand_with_mirrors(tr_paths, tr_labels)
    va_samples = [(p, False) for p in va_paths]
    te_samples = [(p, False) for p in te_paths]

    print(f"Train: {len(tr_paths)} → {len(tr_samples)} (with mirrors)  "
          f"Val: {len(va_paths)}  Test: {len(te_paths)}")

    train_ds = SignDataset(tr_samples, tr_labels_exp, label2idx, cfg, train=True)
    val_ds   = SignDataset(va_samples, va_labels,     label2idx, cfg, train=False)
    test_ds  = SignDataset(te_samples, te_labels,     label2idx, cfg, train=False)

    tr_targets_exp  = [label2idx[l] for l in tr_labels_exp]
    tr_mirror_flags = [s[1] for s in tr_samples]
    sampler = make_weighted_sampler(tr_targets_exp, tr_mirror_flags)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)

    # ── Build V3 model (motion-aware, feat_dim=308) ──
    model = _build_model(cfg, num_classes).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: ASLv3Classifier (motion-aware)  |  Parameters: {total_params:,}")

    # ── Per-class weights ──
    cw = torch.ones(num_classes, device=device)
    for name, idx in label2idx.items():
        if name in TARGET_SIGNS:
            cw[idx] = cfg["target_class_weight"]

    # ── Cluster penalty matrix ──
    cluster_matrix = build_cluster_penalty_matrix(
        classes, CONFUSION_CLUSTERS, penalty=cfg["cluster_penalty"]
    ).to(device)

    criterion = FocalLabelSmoothingCE(
        num_classes            = num_classes,
        smoothing              = cfg["label_smooth"],
        gamma                  = cfg["focal_gamma"],
        class_weights          = cw,
        cluster_penalty_matrix = cluster_matrix,
    )

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    total_steps = cfg["epochs"] * len(train_loader)
    scheduler   = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr           = cfg["max_lr"],
        total_steps      = total_steps,
        pct_start        = 0.3,
        anneal_strategy  = "cos",
        div_factor       = cfg["max_lr"] / cfg["lr"],
        final_div_factor = 1e4,
    )

    best_val_acc = 0.0
    patience_cnt = 0
    best_ckpt    = os.path.join(cfg["save_dir"], "best_model.pt")

    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            criterion, device, cfg["grad_clip"], cfg, num_classes, classes)
        va_loss, va_acc, _, _ = evaluate(
            model, val_loader, criterion, device)

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:3d}/{cfg['epochs']}  "
              f"lr={lr_now:.2e}  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
              f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}")

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            patience_cnt = 0
            torch.save({
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "classes":      classes,
                "label2idx":    label2idx,
                "cfg":          cfg,
                "architecture": "ASLv3Classifier",
                "feat_dim":     cfg.get("feat_dim", FEAT_DIM_MOTION),
            }, best_ckpt)
            print(f"  ✓ Saved best model (val_acc={va_acc:.4f})")
        else:
            patience_cnt += 1
            if patience_cnt >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch}.")
                break

    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model_state"])

    print("\n── Validation confusion analysis ──")
    _, _, va_preds, va_targets = evaluate(model, val_loader, criterion, device)
    print_top_confusions(va_targets, va_preds, classes, top_k=10)

    print("── Standard Test Evaluation (no TTA) ──")
    _, test_acc, preds, targets = evaluate(model, test_loader, criterion, device)
    print(f"Test Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(targets, preds, target_names=classes, digits=4))

    print(f"\n── TTA Test Evaluation ({cfg['tta_n']} views, hand-agnostic) ──")
    tta_acc, tta_preds, tta_targets = evaluate_tta(
        model, test_ds, cfg, device, n_views=cfg["tta_n"])
    print(f"TTA Test Accuracy : {tta_acc:.4f}  "
          f"(vs standard {test_acc:.4f}, Δ={tta_acc - test_acc:+.4f})")
    print("\nTTA Classification Report:")
    print(classification_report(tta_targets, tta_preds, target_names=classes, digits=4))
    print_top_confusions(tta_targets, tta_preds, classes, top_k=10)

    label_path = os.path.join(cfg["save_dir"], "label2idx.json")
    with open(label_path, "w") as f:
        json.dump(label2idx, f, indent=2)
    print(f"\nDone.  Best model : {best_ckpt}")
    print(f"       Label map  : {label_path}")

## 19. Real-Time Inference (Hand-Agnostic, Motion-Aware)

`predict_realtime()` is the function used by a live camera pipeline: it takes
a raw `(60, 155)` buffer of landmarks for the most recent window of frames,
runs it through `_prepare_sequence()` (and optionally an additional mirrored
view, averaging both predictions for hand-agnostic robustness), and returns
either `(predicted_label, confidence)` or `("uncertain", confidence)` if the
top confidence falls below `confidence_threshold`.

In [ ]:
@torch.no_grad()
def predict_realtime(x_np: np.ndarray,
                     model,
                     classes: list,
                     device,
                     use_mirror: bool = True,
                     confidence_threshold: float = 0.4) -> tuple:
    
    model.eval()
    x_np = np.nan_to_num(x_np.astype(np.float32),
                         nan=0.0, posinf=0.0, neginf=0.0)

    orig_seq = _prepare_sequence(x_np.copy(), mirror=False, train=False, cfg={})
    orig  = torch.from_numpy(orig_seq).unsqueeze(0).to(device)
    probs = torch.softmax(model(orig), dim=-1).squeeze(0)

    if use_mirror:
        mir_seq   = _prepare_sequence(x_np.copy(), mirror=True, train=False, cfg={})
        mir       = torch.from_numpy(mir_seq).unsqueeze(0).to(device)
        probs_mir = torch.softmax(model(mir), dim=-1).squeeze(0)
        probs     = (probs + probs_mir) / 2.0

    confidence = probs.max().item()
    label      = classes[probs.argmax().item()]

    if confidence < confidence_threshold:
        return "uncertain", confidence

    return label, confidence

## 20. Batch / File Inference

`predict()` is a convenience entry point for running inference on a single
saved `.npy` file given a checkpoint path: it reloads the model, optionally
runs the full TTA pipeline (`use_tta=True`, guaranteeing a mirrored view) or a
single forward pass, prints the top-5 predicted classes with their
confidences, and returns the top predicted class name.

In [ ]:
def predict(npy_path: str, ckpt_path: str, use_tta: bool = True) -> str:

    ckpt    = torch.load(ckpt_path, map_location="cpu")
    cfg     = ckpt["cfg"]
    classes = ckpt["classes"]
    device  = torch.device("cpu")

    model = _build_model(cfg, len(classes))
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    raw = np.load(npy_path).astype(np.float32)
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)

    if use_tta:
        probs = predict_tta(model, raw, cfg, device, n=cfg.get("tta_n", 5))
    else:
        seq = _prepare_sequence(raw.copy(), mirror=False, train=False, cfg=cfg)
        x   = torch.from_numpy(seq).unsqueeze(0)
        with torch.no_grad():
            probs = torch.softmax(model(x), dim=-1).squeeze(0)

    pred = classes[probs.argmax().item()]

    top5 = probs.topk(min(5, len(classes)))
    print(f"Top predictions {'(TTA+mirror) ' if use_tta else ''}:")
    for score, idx in zip(top5.values, top5.indices):
        print(f"  {classes[idx]:<15} {score.item():.4f}")

    return pred

## 21. Load Model for Inference

`load_model_for_inference()` loads a saved V3 checkpoint (`.pt` file),
reconstructs the model architecture from the checkpoint's stored `cfg`,
restores the trained weights, moves it to the target device, and returns
`(model, classes, device)` ready to be passed into `predict_realtime()` or
`predict()`.

In [ ]:
def load_model_for_inference(ckpt_path: str, device=None):

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt    = torch.load(ckpt_path, map_location=device)
    cfg     = ckpt["cfg"]
    classes = ckpt["classes"]

    model = _build_model(cfg, len(classes))
    model.load_state_dict(ckpt["model_state"])
    model.to(device).eval()

    arch = ckpt.get("architecture", "ASLv3Classifier")
    fd   = ckpt.get("feat_dim", cfg.get("feat_dim", FEAT_DIM_MOTION))
    print(f"Loaded {arch} — {len(classes)} classes, feat_dim={fd}, device={device}")
    return model, classes, device

## 22. Run Training

Kicks off the full pipeline defined above using the `CFG` dictionary from
Cell 2. Before running, double-check `CFG["data_root"]` (folder of one
subfolder per class, each containing `.npy` landmark files of shape `(60,
155)`) and `CFG["save_dir"]` (where the best checkpoint and label map are
written) point to the correct locations for your setup.

**Other usage options** (once the functions above are defined), instead of
running training:

```python
# Real-time inference
model, classes, device = load_model_for_inference("best_model.pt")
label, conf = predict_realtime(frame_buffer, model, classes, device)

# Single-file inference
label = predict("sample.npy", "best_model.pt")            # with TTA
label = predict("sample.npy", "best_model.pt", use_tta=False)
```

> **Note:** checkpoints trained with the OLD 155-dim model are **not**
> compatible with this V3 pipeline — `feat_dim` changed from 155 to 308 and
> the GRU/input-norm weight shapes differ. Retrain from scratch; do not
> attempt to load old `best_model.pt` checkpoints here.

In [ ]:
train(CFG)